In [95]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# d2ski
transfers = pd.read_csv("../data/raw/transfers_d2ski.csv")

# dcaribou
players     = pd.read_csv("../data/raw/players.csv")
appearances = pd.read_csv("../data/raw/appearances.csv")
games       = pd.read_csv("../data/raw/games.csv", 
                           usecols=["game_id", "season"])

print(f"transfers:   {transfers.shape}")
print(f"players:     {players.shape}")
print(f"appearances: {appearances.shape}")
print(f"games:       {games.shape}")

transfers:   (70006, 23)
players:     (37579, 26)
appearances: (1824008, 13)
games:       (80507, 2)


- Every transfer appears twice, "out" and "in", removed the duplicate records keeping only the buying club
- Removed all loans, fee structure is different and market value signal applies differently
- Removed is_free = true, can't compute premiums over market value when the fee is 0
- Removed any transfer_fee_amnts that could be left over
- Removed any market value ammounts less than 0, can't compute 0 in these cases
- Removed broad positions (i.e. midfield, attack, defence), meaningless for our modeling

In [96]:
df = transfers.copy()

df = df[df["dir"] == "in"]
df = df[~df["is_loan"]]
df = df[~df["is_loan_end"]]
df = df[~df["is_free"]]
df = df[df["transfer_fee_amnt"] > 0]
df = df[df["market_val_amnt"] > 0]
df = df[~df["player_pos"].isin(["midfield", "attack", "defence"])]
df = df.drop(columns=["player_nation2"])

df = df.reset_index(drop=True)
print(f"clean transfers: {len(df):,} rows")
print(f"seasons: {sorted(df['season'].unique())}")

clean transfers: 6,097 rows
seasons: [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]


- Convert transfer fee and market value to log transform:
  - fees range from 100k -> 222M, Raw values violate normality assumptions of regression, on the log scale the distribution is roughly symmetric and errors proportional
- Age vs value relationship isn't linear:
  - values may rise through mid-20s and decline after 30, linear age can only capture slope, not a curve
  - adding $age^{2}$ gives the model the ability to fit a parabola.

In [97]:
df["log_transfer_fee"] = np.log(df["transfer_fee_amnt"])
df["log_market_value"] = np.log(df["market_val_amnt"])
df["age_sq"] = df["player_age"] ** 2

Per club, looking at their transfers and keep record of their median fee
- Proxy for their financial ambition and spending patterns.
- Sort by season to ensure that a 2013 transfer has no idea of a clubs future transfer records

In [98]:
df = df.sort_values("season").reset_index(drop=True)

median_fees = []

for idx, row in df.iterrows():
    prior = df[
        (df["counter_team_id"] == row["counter_team_id"]) &
        (df["season"] < row["season"])
    ]
    if len(prior) >= 3:
        median_fees.append(prior["transfer_fee_amnt"].median())
    else:
        median_fees.append(np.nan)

df["buying_club_median_fee"]  = median_fees
df["log_buying_club_median"]  = np.log(
    df["buying_club_median_fee"].clip(lower=1)
)

null_rate = df["buying_club_median_fee"].isna().mean()
print(f"buying_club_median_fee null rate: {null_rate:.1%}")

buying_club_median_fee null rate: 37.2%


In [99]:
players_slim = players[[
    "player_id", "sub_position", "foot",
    "height_in_cm", "country_of_citizenship"
]].drop_duplicates("player_id")

df = df.merge(players_slim, on="player_id", how="left")

print("null rates after player join:")
print(df[["sub_position","foot","height_in_cm",
          "country_of_citizenship"]].isnull().mean().round(3))

null rates after player join:
sub_position              0.023
foot                      0.025
height_in_cm              0.023
country_of_citizenship    0.036
dtype: float64


In [100]:
n_players = df["player_id"].nunique()
n_matched = df[df["sub_position"].notna()]["player_id"].nunique()
print(f"{n_matched}/{n_players} players matched ({n_matched/n_players:.1%})")

4055/4192 players matched (96.7%)


In [101]:
print(f"rows before merge: {len(df)}")
# attach season to each appearance
apps = appearances.merge(games, on="game_id", how="left")
apps = apps.dropna(subset=["season"])
apps["season"] = apps["season"].astype(int)

# aggregate by player and season
season_stats = apps.groupby(["player_id", "season"]).agg(
    goals=("goals", "sum"),
    assists=("assists", "sum"),
    minutes=("minutes_played", "sum"),
    games=("appearance_id", "count")
).reset_index()

# cumulative stats up to but NOT including each season
season_stats = season_stats.sort_values(["player_id", "season"])

season_stats["cum_goals"]   = season_stats.groupby("player_id")["goals"].cumsum()   - season_stats["goals"]
season_stats["cum_assists"] = season_stats.groupby("player_id")["assists"].cumsum() - season_stats["assists"]
season_stats["cum_minutes"] = season_stats.groupby("player_id")["minutes"].cumsum() - season_stats["minutes"]
season_stats["cum_games"]   = season_stats.groupby("player_id")["games"].cumsum()   - season_stats["games"]

# keep only the row for the season matching the transfer
# (which contains cumulative stats BEFORE that season)
df = df.merge(
    season_stats[["player_id", "season",
                  "cum_goals", "cum_assists",
                  "cum_minutes", "cum_games"]],
    on=["player_id", "season"],
    how="left"
)

# per-90 rates
denom = (df["cum_minutes"] / 90).clip(lower=1)
df["goals_per_90"]    = df["cum_goals"]   / denom
df["assists_per_90"]  = df["cum_assists"] / denom
df["minutes_per_game"] = df["cum_minutes"] / df["cum_games"].clip(lower=1)

# fill nulls (players with no prior appearance history)
for col in ["goals_per_90", "assists_per_90", "minutes_per_game"]:
    df[col] = df[col].fillna(0)

df = df.drop(columns=["cum_goals","cum_assists","cum_minutes","cum_games"])

print("null rates after appearance join:")
print(df[["goals_per_90","assists_per_90","minutes_per_game"]].isnull().mean())
print(df[["goals_per_90","assists_per_90","minutes_per_game"]].describe().round(3))
print(f"rows after merge: {len(df)}")

rows before merge: 6097
null rates after appearance join:
goals_per_90        0.0
assists_per_90      0.0
minutes_per_game    0.0
dtype: float64
       goals_per_90  assists_per_90  minutes_per_game
count      6097.000        6097.000          6097.000
mean          0.089           0.063            36.590
std           0.164           0.107            36.871
min           0.000           0.000             0.000
25%           0.000           0.000             0.000
50%           0.000           0.000            33.762
75%           0.105           0.101            74.125
max           1.490           1.508           120.000
rows after merge: 6097


In [102]:
has_stats = (df["minutes_per_game"] > 0).mean()
print(f"fraction with appearance history: {has_stats:.1%}")

# break down by season
print(df.groupby("season")["minutes_per_game"].apply(lambda x: (x > 0).mean()).round(2))

fraction with appearance history: 52.8%
season
2009    0.00
2010    0.00
2011    0.00
2012    0.00
2013    0.56
2014    0.59
2015    0.65
2016    0.68
2017    0.71
2018    0.69
2019    0.67
2020    0.71
2021    0.76
Name: minutes_per_game, dtype: float64


In [103]:
pos_dummies = pd.get_dummies(df["player_pos"], prefix="pos")
df = pd.concat([df, pos_dummies], axis=1)

print(f"position columns: {[c for c in df.columns if c.startswith('pos_')]}")

position columns: ['pos_AM', 'pos_CB', 'pos_CF', 'pos_CM', 'pos_DM', 'pos_GK', 'pos_LB', 'pos_LM', 'pos_LW', 'pos_RB', 'pos_RM', 'pos_RW', 'pos_SS']


In [104]:
null_summary = df.isnull().mean().sort_values(ascending=False)
print(null_summary[null_summary > 0].round(3))
print(f"\nfinal shape: {df.shape}")

buying_club_median_fee    0.372
log_buying_club_median    0.372
country_of_citizenship    0.036
foot                      0.025
sub_position              0.023
height_in_cm              0.023
dtype: float64

final shape: (6097, 47)


In [105]:
df["league_pair"] = df["counter_team_country"] + "_to_" + df["team_country"]

In [106]:
df["has_appearance_history"] = (df["minutes_per_game"] > 0).astype(int)

In [107]:
df["has_buying_club_history"] = df["buying_club_median_fee"].notna().astype(int)
df["buying_club_median_fee"]  = df["buying_club_median_fee"].fillna(
    df["buying_club_median_fee"].median()
)
df["log_buying_club_median"]  = np.log(df["buying_club_median_fee"].clip(lower=1))

In [108]:
df = df.drop(columns=["minutes_per_game"])

In [109]:
train = df[df["season"] <= 2018].reset_index(drop=True)
test  = df[df["season"] >= 2019].reset_index(drop=True)

print(f"train: {len(train):,} rows | seasons {train['season'].min()}–{train['season'].max()}")
print(f"test:  {len(test):,} rows  | seasons {test['season'].min()}–{test['season'].max()}")

df.to_csv("../data/processed/transfers_features.csv", index=False)
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

print("saved.")

train: 4,638 rows | seasons 2009–2018
test:  1,459 rows  | seasons 2019–2021
saved.
